# SurfaceTarget-CRC — phase 04 — patch 4 1a

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1A
#
# DEPMAP 26Q3 FILE IMPORT AND PRELIMINARY VERIFICATION
#
# Required input files:
#   Model.csv
#   CRISPRGeneEffect.csv
#
# Both files must originate from DepMap Public 26Q3.
#
# No dependency analysis performed.
# No existing research results overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import hashlib
import shutil
import json


ROOT = Path("/content/SurfaceTarget-CRC")

UPLOAD_DIR = Path("/content")

DEPMAP_DIR = (
    ROOT / "data/raw/depmap/26Q3"
)

META = ROOT / "data/metadata"

DEPMAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

META.mkdir(
    parents=True,
    exist_ok=True
)


REQUIRED_FILES = [
    "Model.csv",
    "CRISPRGeneEffect.csv"
]


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 4 — PATCH 4.1A")
print("DEPMAP 26Q3 FILE IMPORT")
print("=" * 65)


# ------------------------------------------------------------
# 1. Locate uploaded files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CHECKING UPLOADED FILES")
print("-" * 65)


for filename in REQUIRED_FILES:

    source = UPLOAD_DIR / filename
    destination = DEPMAP_DIR / filename

    if destination.exists():

        print(
            f"\n{filename}: Already in research directory."
        )

        if source.exists():
            print(
                "Additional upload detected; existing "
                "research file will not be overwritten."
            )

    elif source.exists():

        print(f"\nMoving {filename}...")

        shutil.move(
            str(source),
            str(destination)
        )

        print("Move successful.")

    else:

        raise FileNotFoundError(
            f"{filename} not found. "
            "Upload the official 26Q3 file "
            "using Colab's Files panel."
        )


# ------------------------------------------------------------
# 2. SHA256 checksum function
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. Inspect file headers
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("VALIDATING DEPMAP FILE STRUCTURES")
print("-" * 65)


records = []


for filename in REQUIRED_FILES:

    path = DEPMAP_DIR / filename

    print("\nFile:", filename)

    if path.stat().st_size == 0:

        raise RuntimeError(
            f"Empty file detected: {filename}"
        )

    # Read only header and first three rows.
    # The full CRISPR matrix is not loaded.

    preview = pd.read_csv(
        path,
        nrows=3,
        low_memory=False
    )

    columns = (
        preview.columns.astype(str).tolist()
    )

    print(
        "File size (MiB):",
        round(
            path.stat().st_size / (1024**2),
            2
        )
    )

    print("Total columns:", len(columns))

    print("First 8 columns:")
    print(columns[:8])


    if "ModelID" not in columns:

        raise RuntimeError(
            f"ModelID column missing: {filename}"
        )


    if filename == "CRISPRGeneEffect.csv":

        if len(columns) < 1000:

            raise RuntimeError(
                "Unexpectedly few CRISPR gene columns."
            )

        gene_columns = [
            column
            for column in columns
            if column != "ModelID"
        ]

        print(
            "CRISPR gene columns:",
            len(gene_columns)
        )

        print(
            "Example gene identifiers:",
            gene_columns[:5]
        )

    else:

        print(
            "Model metadata columns:",
            len(columns)
        )


    checksum = sha256_file(path)

    print("SHA256:", checksum)

    records.append({

        "release": "DepMap Public 26Q3",

        "filename": filename,

        "local_path": str(path),

        "file_size_bytes":
            path.stat().st_size,

        "total_columns":
            len(columns),

        "header_valid": True,

        "sha256": checksum,

        "source": (
            "User-provided download from "
            "official DepMap portal"
        )

    })


# ------------------------------------------------------------
# 4. Save acquisition records
# ------------------------------------------------------------

audit = pd.DataFrame(records)

audit.to_csv(
    META / "depmap_26q3_import_verification.csv",
    index=False
)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1A",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "declared_release": "DepMap Public 26Q3",

    "download_source": (
        "Official DepMap portal; "
        "manual browser download"
    ),

    "files": records,

    "validation": (
        "Local SHA256 checksums and "
        "preliminary CSV schema checks"
    ),

    "official_checksum_comparison": False,

    "important_note": (
        "Local hashes establish file identity "
        "for reproducibility. Release authenticity "
        "still requires comparison with official "
        "release metadata when available."
    ),

    "dependency_analysis_performed": False,

    "candidate_exclusions": 0

}


with open(
    META / "depmap_26q3_import_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1A SUMMARY")
print("=" * 65)

print("\nRequired files:", len(REQUIRED_FILES))

print(
    "Files passing preliminary checks:",
    len(audit)
)

print("\nDepMap directory:")
print(DEPMAP_DIR)

print("\nAcquisition records: SAVED")

print("Local SHA256 checksums: SAVED")

print("Preliminary CSV validation: PASSED")

print("\nNo dependency calculations performed.")

print("No candidate genes excluded.")

print("No previous research results overwritten.")

print("\nPATCH 4.1A COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 4 — PATCH 4.1A REPAIR
#
# DEPMAP 26Q3 INPUT REPAIR AND VERIFICATION
#
# Fixes:
# 1. Accepts unnamed CRISPR model-ID column
# 2. Detects suspiciously incomplete downloads
# 3. Preserves the previous incomplete file
# 4. Validates CRISPR model identifiers
# 5. Saves reproducibility checksums
#
# No CRISPR dependency analysis performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import hashlib
import shutil
import json


ROOT = Path("/content/SurfaceTarget-CRC")

UPLOAD = Path("/content")

DEPMAP = ROOT / "data/raw/depmap/26Q3"

META = ROOT / "data/metadata"

DEPMAP.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)


print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 4.1A — DEPMAP IMPORT REPAIR")
print("=" * 65)


# ------------------------------------------------------------
# 1. Validate the complete uploaded CRISPR file
# ------------------------------------------------------------

print("\nCHECKING CRISPR FILE SIZE")

uploaded = UPLOAD / "CRISPRGeneEffect.csv"

stored = DEPMAP / "CRISPRGeneEffect.csv"

# The official file was displayed as 497.1 MB.
# This conservative minimum catches the
# previously observed incomplete 54 MiB upload.

MIN_EXPECTED_BYTES = 350_000_000


if uploaded.exists():

    upload_size = uploaded.stat().st_size

    print(
        "New upload size:",
        round(upload_size / 1024**2, 2),
        "MiB"
    )

    if upload_size < MIN_EXPECTED_BYTES:

        raise RuntimeError(
            "The new CRISPR upload appears incomplete. "
            "Wait for the upload to finish and verify "
            "the original file is approximately 497 MB."
        )

    if stored.exists():

        stored_size = stored.stat().st_size

        if stored_size < MIN_EXPECTED_BYTES:

            timestamp = datetime.now(
                timezone.utc
            ).strftime("%Y%m%d_%H%M%S")

            backup = DEPMAP / (
                f"CRISPRGeneEffect_incomplete_{timestamp}.csv"
            )

            stored.rename(backup)

            print(
                "Previous incomplete file preserved:",
                backup.name
            )

        else:

            raise RuntimeError(
                "An apparently complete CRISPR file "
                "already exists in the research directory. "
                "Manual comparison is required before "
                "replacing it."
            )

    shutil.move(
        str(uploaded),
        str(stored)
    )

    print("Complete CRISPR upload moved successfully.")

else:

    print("No new CRISPR upload found.")

    if not stored.exists():

        raise FileNotFoundError(
            "CRISPRGeneEffect.csv is missing."
        )


# ------------------------------------------------------------
# 2. Verify final input files
# ------------------------------------------------------------

model_path = DEPMAP / "Model.csv"

crispr_path = DEPMAP / "CRISPRGeneEffect.csv"

for path in [model_path, crispr_path]:

    if not path.exists():
        raise FileNotFoundError(path)


crispr_bytes = crispr_path.stat().st_size

print("\nFINAL CRISPR FILE SIZE")

print(
    "Size:",
    round(crispr_bytes / 1024**2, 2),
    "MiB"
)

if crispr_bytes < MIN_EXPECTED_BYTES:

    raise RuntimeError(
        "Stored CRISPR file is substantially smaller "
        "than the expected official download."
    )


# ------------------------------------------------------------
# 3. Validate Model.csv
# ------------------------------------------------------------

print("\nVALIDATING MODEL METADATA")

models = pd.read_csv(
    model_path,
    dtype={"ModelID": str},
    low_memory=False
)

if "ModelID" not in models.columns:

    raise RuntimeError(
        "Model.csv does not contain ModelID."
    )

if models["ModelID"].duplicated().any():

    raise RuntimeError(
        "Duplicate ModelIDs in Model.csv."
    )

model_ids = set(
    models["ModelID"].dropna()
)

print("Models in metadata:", len(models))
print("Metadata columns:", len(models.columns))

print("Model metadata validation: PASSED")


# ------------------------------------------------------------
# 4. Validate CRISPR header
# ------------------------------------------------------------

print("\nVALIDATING CRISPR GENE-EFFECT DATA")

header = pd.read_csv(
    crispr_path,
    nrows=0
).columns.tolist()

print("Total CRISPR columns:", len(header))

print("Original first column:", header[0])

print("First gene columns:", header[1:6])


if len(header) < 15000:

    raise RuntimeError(
        "Unexpectedly few CRISPR gene columns."
    )


if header[0] not in [
    "ModelID",
    "Unnamed: 0"
]:

    raise RuntimeError(
        "Unrecognized CRISPR model identifier column."
    )


# The first column is treated as ModelID
# regardless of whether its header is blank.
#
# The original source file is not rewritten.

print("\nModel identifier handling: CORRECTED")


# ------------------------------------------------------------
# 5. Validate all CRISPR model identifiers
# ------------------------------------------------------------

print("\nREADING CRISPR MODEL IDENTIFIERS")

# Read only the first column into memory.
# Pandas parses the complete CSV while retaining
# only the model identifier column.

crispr_index = pd.read_csv(
    crispr_path,
    usecols=[0],
    dtype=str
)

crispr_index.columns = ["ModelID"]

crispr_index["ModelID"] = (
    crispr_index["ModelID"]
    .astype(str)
    .str.strip()
)

crispr_ids = crispr_index["ModelID"]

if crispr_ids.duplicated().any():

    raise RuntimeError(
        "Duplicate CRISPR model identifiers."
    )

valid_format = (
    crispr_ids.str.fullmatch(
        r"ACH-\d+"
    )
)

if not valid_format.all():

    raise RuntimeError(
        "Unexpected CRISPR model identifier format."
    )


matched = crispr_ids.isin(
    model_ids
)

print("CRISPR model rows:", len(crispr_ids))

print(
    "Models matching Model.csv:",
    int(matched.sum())
)

print(
    "Models without metadata match:",
    int((~matched).sum())
)


if not matched.all():

    raise RuntimeError(
        "Some CRISPR model IDs are missing "
        "from the supplied Model.csv. "
        "Confirm both files are from 26Q3."
    )


print("\nCRISPR model identifier validation: PASSED")


# ------------------------------------------------------------
# 6. Save checksums
# ------------------------------------------------------------

print("\nCALCULATING SOURCE CHECKSUMS")


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


records = []

for path in [model_path, crispr_path]:

    checksum = sha256_file(path)

    records.append({

        "release": "DepMap Public 26Q3",

        "filename": path.name,

        "local_path": str(path),

        "file_size_bytes":
            path.stat().st_size,

        "sha256": checksum,

        "header_valid": True,

        "source": "Official DepMap browser download"
    })

    print(path.name)
    print("SHA256:", checksum)


audit = pd.DataFrame(records)

audit.to_csv(
    META / "depmap_26q3_import_verification.csv",
    index=False
)


provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "4.1A-repair",

    "release": "DepMap Public 26Q3",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "crispr_model_column_original": header[0],

    "crispr_model_column_interpretation": "ModelID",

    "crispr_model_rows": int(len(crispr_ids)),

    "crispr_gene_columns": int(len(header) - 1),

    "matched_model_ids": int(matched.sum()),

    "source_files": records,

    "official_hash_comparison_performed": False,

    "dependency_analysis_performed": False,

    "original_crispr_file_modified": False

}

with open(
    META / "depmap_26q3_import_provenance.json",
    "w"
) as handle:

    json.dump(
        provenance,
        handle,
        indent=4
    )


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 4.1A — REPAIR SUMMARY")
print("=" * 65)

print("\nDepMap release: 26Q3")

print("Files verified: 2")

print(
    "CRISPR model rows:",
    len(crispr_ids)
)

print(
    "CRISPR gene columns:",
    len(header) - 1
)

print(
    "Models matched to metadata:",
    int(matched.sum())
)

print("\nModel identifier fix: APPLIED")

print("File size validation: PASSED")

print("Model correspondence: PASSED")

print("SHA256 checksums: SAVED")

print("Source provenance: SAVED")

print("\nNo dependency analysis performed.")

print("\nPATCH 4.1A COMPLETE")

print("=" * 65)